# Препроцессинг и feature engineering

Загружаем, чистим, делаем новые фичи, готовим pipeline для разных семейств моделей. Всё, что делается тут, оформлено в `src/data.py` и `src/features.py` - здесь просто прогоняем и смотрим.

In [1]:
import sys
sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')

from src.data import load_raw, clean
from src.features import add_features, feature_columns, NUMERIC_COLS, LOW_CARD_CAT, HIGH_CARD_CAT

### Загрузка и чистка

In [2]:
raw = load_raw('../data/hotels.csv')
print('raw:', raw.shape)
df = clean(raw)
print('after clean:', df.shape)

raw: (119390, 32)
after clean: (86970, 30)


Ушло около 32 тысяч строк - это в основном дубликаты, плюс немного строк с нулевыми гостями и странным adr.

In [3]:
df['is_canceled'].mean()

np.float64(0.2730826721858112)

Баланс немного поплыл (около 27% отмен) - дубликаты в основном на стороне "не отменили", что разумно.

### Новые фичи

In [4]:
df = add_features(df)
df[['total_nights','total_guests','has_children','is_family','room_changed','season','arrival_weekday','adr_per_person']].head()

,total_nights,total_guests,has_children,is_family,room_changed,season,arrival_weekday,adr_per_person
0,0,2,0,0,0,summer,2,0.0
1,0,2,0,0,0,summer,2,0.0
2,1,1,0,0,1,summer,2,75.0
3,1,1,0,0,0,summer,2,75.0
4,2,2,0,0,0,summer,2,49.0


Краткий смысл новых фич:
- `total_nights`, `total_guests` - агрегаты, чаще удобнее чем их компоненты
- `has_children`, `is_family` - семейные брони ведут себя иначе
- `room_changed` - по EDA видно, что это связано с отменой
- `arrival_weekday`, `season` - сезонность в одном числе
- `adr_per_person` - нормируем цену, большая группа платит меньше per capita

### Разделение X / y

In [5]:
cols = feature_columns()
X = df[cols].copy()
y = df['is_canceled'].astype(int)
X.shape, y.shape

((86970, 37), (86970,))

### train/test split

In [6]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(X_train.shape, X_test.shape)
print('train cancel rate:', y_train.mean().round(3), '| test:', y_test.mean().round(3))

(69576, 37) (17394, 37)
train cancel rate: 0.273 | test: 0.273


Стратифицировали по таргету, доли совпадают. random_state=42 чтобы воспроизводить.

### Pipeline для линейных моделей (OHE + scale + target encoding для country)

In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from category_encoders import TargetEncoder

def make_linear_preprocessor():
    num = StandardScaler()
    ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    te = TargetEncoder(cols=HIGH_CARD_CAT, smoothing=10)
    return ColumnTransformer([
        ('num', num, NUMERIC_COLS),
        ('cat', ohe, LOW_CARD_CAT),
        ('te', te, HIGH_CARD_CAT),
    ], remainder='drop')

pre_linear = make_linear_preprocessor()
Xt = pre_linear.fit_transform(X_train, y_train)
print('linear feature matrix:', Xt.shape)

linear feature matrix: (69576, 79)


### Pipeline для деревьев/бустингов (OrdinalEncoder)

In [8]:
from sklearn.preprocessing import OrdinalEncoder

def make_tree_preprocessor():
    oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    return ColumnTransformer([
        ('num', 'passthrough', NUMERIC_COLS),
        ('cat', oe, LOW_CARD_CAT + HIGH_CARD_CAT),
    ], remainder='drop')

pre_tree = make_tree_preprocessor()
Xt2 = pre_tree.fit_transform(X_train)
print('tree feature matrix:', Xt2.shape)

tree feature matrix: (69576, 37)


### Сохраняем подготовленные данные

Чтобы следующие ноутбуки не перегоняли всё с нуля.

In [9]:
import joblib, os
os.makedirs('../models', exist_ok=True)
joblib.dump({
    'X_train': X_train, 'X_test': X_test,
    'y_train': y_train, 'y_test': y_test,
}, '../models/_split.pkl')
print('saved split')

saved split


Готово. Дальше - бейзлайн.